In [25]:
import pandas as pd

df = pd.read_csv("heart.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (1025, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [26]:
df.info()#Dataset Information

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


In [27]:
#Separate Features and Target
X = df.drop("target", axis=1)

y = df["target"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (1025, 13)
Target Shape: (1025,)


In [28]:
#Split Data into Training and Testing
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (820, 13)
Testing Shape: (205, 13)


In [29]:
#Create test_data.csv
test_df = X_test.copy()

test_df["target"] = y_test

test_df.to_csv(
    "test_data.csv",
    index=False
)

print("test_data.csv created successfully")

test_data.csv created successfully


In [30]:
#Verify File Created
import os

print(os.listdir())

['.config', 'heart.csv', 'test_data.csv', 'model', 'model_comparison.csv', 'sample_data']


In [31]:
#Train All 5 Models.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

# Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# KNN
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

# Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)

# SVM
svm = SVC(probability=True)
svm.fit(X_train, y_train)

print("All 5 Models Trained Successfully")

All 5 Models Trained Successfully


In [32]:
#Calculate All Required Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score
)

import pandas as pd

results = []

models = {
    "Logistic Regression": lr,
    "Decision Tree": dt,
    "KNN": knn,
    "Naive Bayes": nb,
    "SVM": svm
}

for name, model in models.items():

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        matthews_corrcoef(y_test, y_pred)
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "AUC",
        "Precision",
        "Recall",
        "F1",
        "MCC"
    ]
)

results_df

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.814634,0.929905,0.763780,0.923810,0.836207,0.642240
1,Decision Tree,0.985366,0.985714,1.000000,0.971429,0.985507,0.971151
2,KNN,0.697561,0.833524,0.712871,0.685714,0.699029,0.395639
3,Naive Bayes,0.829268,0.904286,0.807018,0.876190,0.840183,0.660163
4,SVM,0.717073,0.747714,0.697479,0.790476,0.741071,0.436035


In [33]:
#save the Comparison Table
results_df.to_csv(
    "model_comparison.csv",
    index=False
)

print("model_comparison.csv saved")

model_comparison.csv saved


In [34]:
#Find the Best Model
results_df.sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
1,Decision Tree,0.985366,0.985714,1.000000,0.971429,0.985507,0.971151
3,Naive Bayes,0.829268,0.904286,0.807018,0.876190,0.840183,0.660163
0,Logistic Regression,0.814634,0.929905,0.763780,0.923810,0.836207,0.642240
4,SVM,0.717073,0.747714,0.697479,0.790476,0.741071,0.436035
2,KNN,0.697561,0.833524,0.712871,0.685714,0.699029,0.395639


In [35]:
#save all models(.pkl Files)
import os
import joblib

# Create model folder
os.makedirs("model", exist_ok=True)

# Save models
joblib.dump(lr, "model/logistic.pkl")
joblib.dump(dt, "model/decision_tree.pkl")
joblib.dump(knn, "model/knn.pkl")
joblib.dump(nb, "model/naive_bayes.pkl")
joblib.dump(svm, "model/svm.pkl")

print("All models saved successfully")

All models saved successfully


In [36]:
import os

print(os.listdir("model"))

['svm.pkl', 'knn.pkl', 'decision_tree.pkl', 'logistic.pkl', 'naive_bayes.pkl']
